# 🔍 Binary Search — Master Guide (Sean Edition)

---

**Mental model:** Binary search is a **book-index lookup**. You open the index to the middle page. If your word comes before the middle, tear off the right half — it can't be there. Repeat on the left half. Each guess cuts the remaining search space in half. 1 billion elements? Thirty comparisons max. That's O(log n) — the most powerful trick in algorithm design.

---

## Table of Contents

1. [Visual Model — How Binary Search Works](#1)
2. [Setup — The Binary Search Template](#2)
3. [API Quick Reference](#3)
4. [Decision Map — Which Template to Use](#4)
5. [Pattern 1 — Classic Binary Search (LC 704)](#5)
6. [Pattern 2 — Search in Rotated Sorted Array (LC 33)](#6)
7. [Pattern 3 — Find Minimum in Rotated Array (LC 153)](#7)
8. [Pattern 4 — Search a 2D Matrix (LC 74)](#8)
9. [Pattern 5 — Search on Answer Space (LC 875)](#9)
10. [Full Decision Map — All 5 Patterns](#10)
11. [Cheat Sheet + Summary](#11)

<a id='1'></a>

## 1. 🗺️ Visual Model — How Binary Search Works

---

### The Three Cases

```
Array:  [ -1,  0,  3,  5,  9, 12 ]   target = 9
Index:    0    1   2   3   4   5
          ↑            ↑        ↑
         lo           mid      hi

  nums[mid]=5, target=9 → target > mid → lo = mid+1

Array:  [ -1,  0,  3,  5,  9, 12 ]   target = 9
Index:    0    1   2   3   4   5
                             ↑  ↑
                            lo  hi
                            mid

  nums[mid]=9, target=9 → FOUND at index 4
```

```
CASE 1: target < nums[mid]   → hi = mid - 1   (discard right half)
CASE 2: target > nums[mid]   → lo = mid + 1   (discard left half)
CASE 3: target == nums[mid]  → return mid      (done)
```

---

### How the Search Space Halves

```
Step 0:  [______________________] 16 elements
Step 1:  [__________]             8 elements  (half discarded)
Step 2:  [_____]                  4 elements  (half discarded)
Step 3:  [__]                     2 elements  (half discarded)
Step 4:  [_]                      1 element   → found or not found

log2(1,000,000,000) ≈ 30   →   30 comparisons for 1 billion elements
```

---

### Off-by-One Landscape

```
TEMPLATE A — Exact match (lo <= hi):          TEMPLATE B — Boundary (lo < hi):

  lo=0, hi=n-1                                  lo=0, hi=n  (hi is PAST end)
  while lo <= hi:                               while lo < hi:
      mid = lo + (hi-lo)//2                         mid = lo + (hi-lo)//2
      ...                                           ...
  return -1  (not found)                        return lo   (the boundary)

  Terminates when lo > hi.                      Terminates when lo == hi.
  Use for: exact match.                         Use for: first/last occurrence,
                                                         answer space.

OVERFLOW-SAFE MID:
  mid = lo + (hi - lo) // 2   ✅   same as (lo+hi)//2 but no int overflow
  mid = (lo + hi) // 2        ⚠️   overflows in C/C++ for large indices
```

---

### Left Boundary vs Right Boundary

```
Array: [ 1, 2, 2, 2, 3 ]
Index:   0  1  2  3  4

bisect_left(arr, 2)   → 1   (leftmost position where 2 can be inserted)
bisect_right(arr, 2)  → 4   (rightmost position where 2 can be inserted)

LEFT BOUNDARY:   hi = mid       (when feasible, try to go lower)
RIGHT BOUNDARY:  lo = mid + 1   (when feasible, keep going right)
```

---

### Why This Matters

```
Linear scan (O(n)):      1,000,000,000 ops   → seconds
Binary search (O(log n)):           30 ops   → nanoseconds

Prerequisite: data must be SORTED (or have a monotone predicate).
```

<a id='2'></a>

## 2. 🔧 Setup — The Binary Search Template

In [ ]:
# ─────────────────────────────────────────────────────────────
# UNIVERSAL BINARY SEARCH TEMPLATES
# ─────────────────────────────────────────────────────────────


# ── TEMPLATE 1: STANDARD (exact match) ───────────────────────
def bs_standard(nums, target):
    """Find exact target index. Returns -1 if not found."""
    lo, hi = 0, len(nums) - 1          # both ends inclusive
    while lo <= hi:                     # equal is valid: 1-element range
        mid = lo + (hi - lo) // 2      # overflow-safe mid
        if nums[mid] == target:         # exact match
            return mid
        elif nums[mid] < target:        # target is to the right
            lo = mid + 1               # discard left half
        else:                           # target is to the left
            hi = mid - 1               # discard right half
    return -1                           # loop exits → not found


# ── TEMPLATE 2: LEFT BOUNDARY (first occurrence / minimize valid) ──
def bs_left_boundary(nums, target):
    """Find leftmost index where nums[i] == target. Returns -1 if not found."""
    lo, hi = 0, len(nums)              # hi = len (one past end)
    while lo < hi:                     # strict: stops when lo == hi
        mid = lo + (hi - lo) // 2
        if nums[mid] < target:          # mid is too small → move right
            lo = mid + 1
        else:                           # nums[mid] >= target → could be answer, pull hi in
            hi = mid
    # lo == hi is now the leftmost index where nums[i] >= target
    if lo < len(nums) and nums[lo] == target:
        return lo
    return -1


# ── TEMPLATE 3: RIGHT BOUNDARY (last occurrence / maximize valid) ──
def bs_right_boundary(nums, target):
    """Find rightmost index where nums[i] == target. Returns -1 if not found."""
    lo, hi = 0, len(nums)              # hi = len (one past end)
    while lo < hi:
        mid = lo + (hi - lo) // 2
        if nums[mid] <= target:         # mid could still be answer → move right
            lo = mid + 1
        else:
            hi = mid
    # lo - 1 is the rightmost index where nums[i] <= target
    if lo > 0 and nums[lo - 1] == target:
        return lo - 1
    return -1


# ── TEMPLATE 4: ANSWER SPACE (binary search on values) ───────
def bs_answer_space(lo, hi, feasible_fn):
    """Find minimum value in [lo, hi] where feasible_fn(mid) is True."""
    while lo < hi:                     # stops when lo == hi → that's the answer
        mid = lo + (hi - lo) // 2
        if feasible_fn(mid):            # mid works → maybe smaller works too
            hi = mid                   # pull hi down to mid
        else:
            lo = mid + 1               # mid too small → go right
    return lo                          # lo == hi is the minimum valid value


# ── DEMO ──────────────────────────────────────────────────────
sample = [-1, 0, 3, 5, 9, 12]

print("=== STANDARD ===")
print(f"search 9  → index {bs_standard(sample, 9)}")    # 4
print(f"search -1 → index {bs_standard(sample, -1)}")   # 0
print(f"search 2  → index {bs_standard(sample, 2)}")    # -1

dupes = [1, 2, 2, 2, 3]
print("\n=== LEFT BOUNDARY ===")
print(f"left(2)  → {bs_left_boundary(dupes, 2)}")       # 1
print("\n=== RIGHT BOUNDARY ===")
print(f"right(2) → {bs_right_boundary(dupes, 2)}")      # 3

print("\n=== ANSWER SPACE ===")
# find minimum x in [1,10] where x*x >= 20
result = bs_answer_space(1, 10, lambda x: x * x >= 20)
print(f"min x where x²>=20 → {result}")                 # 5 (5²=25)

<a id='3'></a>

## 3. 📋 API Quick Reference

```
OPERATION                     COMPLEXITY   USE CASE
──────────────────────────────────────────────────────────────
Standard binary search         O(log n)     Exact match in sorted array
bisect_left(arr, x)            O(log n)     Leftmost position to insert x
bisect_right(arr, x)           O(log n)     Rightmost position to insert x
bisect.insort(arr, x)          O(n)         Insert x maintaining sort (slow insert)
Binary search on rotated        O(log n)     One half always sorted — identify which
Binary search on answer space   O(log(R-L)) Predicate: can_do(mid)? find boundary
──────────────────────────────────────────────────────────────
THE THREE TEMPLATES:
1. EXACT MATCH:    lo, hi = 0, len-1;  while lo <= hi
2. LEFT BOUNDARY:  lo, hi = 0, len;    while lo < hi;  return lo
3. ANSWER SPACE:   lo, hi = min, max;  while lo < hi;  minimize valid mid
──────────────────────────────────────────────────────────────
THINGS YOU DO NOT DO:
❌  mid = (lo + hi) // 2  →  integer overflow in other languages; use lo + (hi-lo)//2
❌  Binary search on unsorted data — it will silently give wrong answers
❌  Off-by-one: while lo < hi with lo = mid (infinite loop!) — always check termination
```

In [ ]:
import bisect
import math

# ── bisect_left and bisect_right ──────────────────────────────
arr = [1, 2, 2, 2, 3, 5, 8]

print("=== bisect demo ===")
print(f"arr = {arr}")
print(f"bisect_left(arr, 2)  = {bisect.bisect_left(arr, 2)}")   # 1
print(f"bisect_right(arr, 2) = {bisect.bisect_right(arr, 2)}")  # 4
print(f"bisect_left(arr, 4)  = {bisect.bisect_left(arr, 4)}")   # 5 (insertion point)

# ── TRACE: standard template step by step ─────────────────────
def bs_traced(nums, target):
    """Standard binary search with step-by-step trace printed."""
    lo, hi = 0, len(nums) - 1
    step = 0
    while lo <= hi:
        mid = lo + (hi - lo) // 2
        print(f"  step {step}: lo={lo} mid={mid} hi={hi}  nums[mid]={nums[mid]}")
        if nums[mid] == target:
            print(f"  → FOUND at index {mid}")
            return mid
        elif nums[mid] < target:
            lo = mid + 1               # move right
        else:
            hi = mid - 1               # move left
        step += 1
    print("  → NOT FOUND")
    return -1

print("\n=== TRACE: search 9 in [-1,0,3,5,9,12] ===")
bs_traced([-1, 0, 3, 5, 9, 12], 9)

print("\n=== TRACE: search 2 in [-1,0,3,5,9,12] (not found) ===")
bs_traced([-1, 0, 3, 5, 9, 12], 2)

# ── left boundary trace ───────────────────────────────────────
def bs_left_traced(nums, target):
    """Left boundary search with trace."""
    lo, hi = 0, len(nums)              # hi = len (open right boundary)
    step = 0
    while lo < hi:
        mid = lo + (hi - lo) // 2
        print(f"  step {step}: lo={lo} mid={mid} hi={hi}  nums[mid]={nums[mid]}")
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid                   # shrink right, not mid-1
        step += 1
    print(f"  → left boundary at {lo}")
    return lo

print("\n=== TRACE: left boundary of 2 in [1,2,2,2,3] ===")
bs_left_traced([1, 2, 2, 2, 3], 2)

<a id='4'></a>

## 4. 🧭 Decision Map — Which Template to Use

```
SIGNAL IN THE PROBLEM                   WHICH TEMPLATE
──────────────────────────────────────────────────────────────────────
"find target in sorted array"           standard: lo<=hi, exact match
"find first/leftmost occurrence"        left boundary: lo < hi, return lo
"find last/rightmost occurrence"        right boundary: lo < hi, return lo-1
"sorted but rotated"                    LC 33 pattern: identify sorted half
"find minimum in rotated"               LC 153: min always in unsorted half
"search in 2D matrix (row sorted)"      treat as 1D: row=mid//n, col=mid%n
"minimize max / maximize min"           answer space: binary search on values
"'if we could do X with Y, can we...'"  predicate binary search on answer
```

<a id='5'></a>

## 5. 🔍 Pattern 1 — Classic Binary Search (LC 704)

---

**PROBLEM:** Given a sorted array of integers and a target, return the index of the target. Return -1 if not found.

**TRICK:** Two pointers `lo` and `hi` converge toward the target. Each step eliminates half the remaining elements. No target, no match.

---

**SLOW MOTION TRACE** — `nums=[-1,0,3,5,9,12]`, `target=9`

```
Initial:  lo=0   hi=5

Step 0:   mid = 0 + (5-0)//2 = 2   nums[2]=3   3 < 9 → lo = mid+1 = 3
          [ -1,  0,  3, |  5,  9, 12 ]
                         ↑lo         ↑hi

Step 1:   mid = 3 + (5-3)//2 = 4   nums[4]=9   9 == 9 → FOUND index 4
          [ -1,  0,  3,   5,  9, 12 ]
                         ↑lo  ↑mid  ↑hi
```

---

**KEY INSIGHT:** `lo <= hi` allows the single-element case. When `lo > hi`, the space is exhausted — target is absent.

**TIME:** O(log n) — halve search space each step  
**SPACE:** O(1) — two pointers only

In [ ]:
# LC 704 — Classic Binary Search
# Time: O(log n)   Space: O(1)

def search(nums, target):
    """
    Find target in sorted array. Return index, -1 if not found.
    Time: O(log n)   Space: O(1)

    Slow motion trace (nums=[-1,0,3,5,9,12], target=9):
      lo=0, hi=5
      step 0: mid=2  nums[2]=3  3<9  → lo=3
      step 1: mid=4  nums[4]=9  9==9 → return 4
    """
    lo, hi = 0, len(nums) - 1          # WHY -1: hi is inclusive, last valid index

    while lo <= hi:                     # WHY <=: single element (lo==hi) must be checked
        mid = lo + (hi - lo) // 2      # WHY this form: avoids int overflow vs (lo+hi)//2

        if nums[mid] == target:         # exact match — done
            return mid
        elif nums[mid] < target:        # WHY +1: mid already checked, skip it
            lo = mid + 1
        else:                           # WHY -1: mid already checked, skip it
            hi = mid - 1

    return -1                           # lo > hi: search space exhausted


# ── test harness ──────────────────────────────────────────────
def test_harness(fn, cases):
    """Run test cases. cases = list of (args_tuple, expected)."""
    passed = 0
    for args, expected in cases:
        result = fn(*args)
        status = "PASS" if result == expected else "FAIL"
        if status == "FAIL":
            print(f"  {status}: {fn.__name__}{args} → got {result}, expected {expected}")
        passed += (status == "PASS")
    print(f"  {passed}/{len(cases)} passed")


cases_704 = [
    # (args),                         expected
    (([-1, 0, 3, 5, 9, 12], 9),        4),   # middle target
    (([-1, 0, 3, 5, 9, 12], -1),       0),   # first element
    (([-1, 0, 3, 5, 9, 12], 12),       5),   # last element
    (([-1, 0, 3, 5, 9, 12], 2),       -1),   # not found
    (([5], 5),                          0),   # single element found
    (([5], 3),                         -1),   # single element not found
    (([1, 3], 3),                       1),   # two elements, right
    (([1, 3], 1),                       0),   # two elements, left
]

test_harness(search, cases_704)
print("search defined.")

<a id='6'></a>

## 6. 🔄 Pattern 2 — Search in Rotated Sorted Array (LC 33)

---

**PROBLEM:** A sorted array was rotated at some pivot. Given a target, find its index. Return -1 if not found. Assume no duplicates.

**TRICK:** One half is ALWAYS sorted. Check if the target falls in the sorted half. If yes, search there. If no, search the other half.

---

**SLOW MOTION TRACE** — `nums=[4,5,6,7,0,1,2]`, `target=0`

```
Initial:  lo=0   hi=6

Step 0:   mid=3  nums[3]=7
          Left half [4,5,6,7] is sorted (nums[lo]=4 <= nums[mid]=7)
          Is target 0 in [4..7]?  NO → search right half
          lo = mid+1 = 4

Step 1:   mid=5  nums[5]=1
          Left half [0,1] is sorted (nums[lo]=0 <= nums[mid]=1)
          Is target 0 in [0..1]?  YES → search left half
          hi = mid-1 = 4

Step 2:   mid=4  nums[4]=0  0==0 → FOUND index 4
```

---

**KEY INSIGHT:** "One half is always clean." At every step, one of the two halves is a plain sorted subarray. Identify which, check if target belongs there, recurse.

**TIME:** O(log n)  
**SPACE:** O(1)

In [ ]:
# LC 33 — Search in Rotated Sorted Array
# Time: O(log n)   Space: O(1)

def search_rotated(nums, target):
    """
    Find target in a rotated sorted array. Return index, -1 if not found.
    Time: O(log n)   Space: O(1)

    Slow motion trace (nums=[4,5,6,7,0,1,2], target=0):
      step 0: mid=3 nums[3]=7  left[4..7] sorted, 0 not in [4..7] → lo=4
      step 1: mid=5 nums[5]=1  left[0..1] sorted, 0 in [0..1]     → hi=4
      step 2: mid=4 nums[4]=0  0==0 → return 4
    """
    lo, hi = 0, len(nums) - 1

    while lo <= hi:
        mid = lo + (hi - lo) // 2

        if nums[mid] == target:             # exact match
            return mid

        # WHY: check if left half [lo..mid] is the sorted half
        if nums[lo] <= nums[mid]:           # left half is sorted
            # WHY: target must be strictly inside the sorted range to belong here
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1               # target is in sorted left half
            else:
                lo = mid + 1               # target is in right half (possibly rotated)
        else:                               # right half [mid..hi] is sorted
            # WHY: same logic flipped for right half
            if nums[mid] < target <= nums[hi]:
                lo = mid + 1               # target is in sorted right half
            else:
                hi = mid - 1               # target is in left half (possibly rotated)

    return -1


cases_33 = [
    (([4, 5, 6, 7, 0, 1, 2], 0),   4),   # target in rotated portion
    (([4, 5, 6, 7, 0, 1, 2], 3),  -1),   # not found
    (([4, 5, 6, 7, 0, 1, 2], 4),   0),   # first element
    (([4, 5, 6, 7, 0, 1, 2], 2),   6),   # last element
    (([1], 0),                     -1),   # single element not found
    (([1], 1),                      0),   # single element found
    (([6, 7, 1, 2, 3, 4, 5], 1),   2),   # rotated at start
    (([3, 1], 1),                   1),   # two elements rotated
]

test_harness(search_rotated, cases_33)
print("search_rotated defined.")

<a id='7'></a>

## 7. ⬇️ Pattern 3 — Find Minimum in Rotated Array (LC 153)

---

**PROBLEM:** A sorted array was rotated at an unknown pivot. Find the minimum element. No duplicates.

**TRICK:** The minimum is always in the "unsorted" half. If `nums[mid] > nums[hi]`, the rotation point is in the right half — min is there. Otherwise min is in the left half (or is mid itself).

---

**SLOW MOTION TRACE** — `nums=[3,4,5,1,2]`

```
Initial:  lo=0   hi=4

Step 0:   mid=2  nums[2]=5  nums[hi]=2
          5 > 2 → min is in RIGHT half  → lo = mid+1 = 3
          [ 3,  4,  5, | 1,  2 ]
                         ↑lo    ↑hi

Step 1:   mid=3  nums[3]=1  nums[hi]=2
          1 < 2 → min is in LEFT half (could be mid)  → hi = mid = 3
          [ 3,  4,  5,  1,  2 ]
                       ↑lo=hi

Terminate: lo == hi == 3  → return nums[3] = 1
```

---

**KEY INSIGHT:** Compare `nums[mid]` to `nums[hi]` (not `nums[lo]`). If mid > hi-side, rotation pivot is right of mid — go right. Else go left (min is at mid or left of mid).

**TIME:** O(log n)  
**SPACE:** O(1)

In [ ]:
# LC 153 — Find Minimum in Rotated Sorted Array
# Time: O(log n)   Space: O(1)

def find_min(nums):
    """
    Find minimum in a rotated sorted array (no duplicates).
    Time: O(log n)   Space: O(1)

    Slow motion trace (nums=[3,4,5,1,2]):
      step 0: lo=0 hi=4 mid=2 nums[2]=5 > nums[4]=2 → lo=3
      step 1: lo=3 hi=4 mid=3 nums[3]=1 < nums[4]=2 → hi=3
      terminate: lo==hi==3, return nums[3]=1
    """
    lo, hi = 0, len(nums) - 1

    while lo < hi:                          # WHY <: stop when lo==hi (single candidate)
        mid = lo + (hi - lo) // 2

        # WHY compare to nums[hi]: hi is always the "right edge" of current window
        if nums[mid] > nums[hi]:            # mid is in the higher (left) portion
            # WHY mid+1: nums[mid] is NOT the minimum (it's too big)
            lo = mid + 1                    # min must be right of mid
        else:                               # nums[mid] <= nums[hi]
            # WHY hi=mid not mid-1: nums[mid] could BE the minimum
            hi = mid                        # min is at mid or left of mid

    return nums[lo]                         # lo == hi: the minimum element


cases_153 = [
    (([3, 4, 5, 1, 2],),     1),   # rotated in middle
    (([4, 5, 6, 7, 0, 1, 2],), 0), # rotated near end
    (([11, 13, 15, 17],),    11),   # not rotated (min at start)
    (([2, 1],),               1),   # two elements rotated
    (([1],),                  1),   # single element
    (([1, 2],),               1),   # two elements, no rotation
    (([5, 1, 2, 3, 4],),      1),   # rotated at position 1
]

# WHY adapt test_harness: find_min takes a single list, not (list, target)
def test_harness_1arg(fn, cases):
    """Run test cases where fn takes a single argument tuple."""
    passed = 0
    for args, expected in cases:
        result = fn(*args)
        status = "PASS" if result == expected else "FAIL"
        if status == "FAIL":
            print(f"  {status}: {fn.__name__}{args} → got {result}, expected {expected}")
        passed += (status == "PASS")
    print(f"  {passed}/{len(cases)} passed")


test_harness_1arg(find_min, cases_153)
print("find_min defined.")

<a id='8'></a>

## 8. 📐 Pattern 4 — Search a 2D Matrix (LC 74)

---

**PROBLEM:** An `m x n` matrix where each row is sorted and the first element of each row is greater than the last element of the previous row. Find target. Return True/False.

**TRICK:** Treat the entire matrix as a 1D sorted array of size `m*n`. Map 1D index to 2D: `row = idx // n`, `col = idx % n`. One binary search over `[0, m*n - 1]`.

---

**SLOW MOTION TRACE** — 3×4 matrix, `target=7`

```
Matrix:   [  1,  3,  5,  7 ]   row 0
          [ 10, 11, 16, 20 ]   row 1
          [ 23, 30, 34, 60 ]   row 2

Treat as 1D: [1,3,5,7,10,11,16,20,23,30,34,60]  (indices 0..11)

lo=0, hi=11
Step 0:  mid=5  idx 5 → row=5//4=1, col=5%4=1  → matrix[1][1]=11  11>7 → hi=4
Step 1:  mid=2  idx 2 → row=2//4=0, col=2%4=2  → matrix[0][2]=5   5<7  → lo=3
Step 2:  mid=3  idx 3 → row=3//4=0, col=3%4=3  → matrix[0][3]=7   7==7 → True
```

---

**KEY INSIGHT:** The row-sorted + first>last-of-prev constraint means the matrix is globally sorted. Map to 1D and run standard binary search.

**TIME:** O(log(m × n))  
**SPACE:** O(1)

In [ ]:
# LC 74 — Search a 2D Matrix
# Time: O(log(m*n))   Space: O(1)

def search_matrix(matrix, target):
    """
    Search target in row-sorted matrix where first elem of row > last of prev row.
    Time: O(log(m*n))   Space: O(1)

    Slow motion trace (3x4 matrix, target=7):
      Treat as 1D [1,3,5,7,10,11,16,20,23,30,34,60]
      step 0: mid=5  → [1][1]=11  11>7 → hi=4
      step 1: mid=2  → [0][2]=5   5<7  → lo=3
      step 2: mid=3  → [0][3]=7   7==7 → True
    """
    m = len(matrix)                         # number of rows
    n = len(matrix[0])                      # number of columns

    lo, hi = 0, m * n - 1                  # treat as 1D array of size m*n

    while lo <= hi:
        mid = lo + (hi - lo) // 2

        # WHY //n and %n: converts flat 1D index back to 2D row, col
        row = mid // n
        col = mid % n
        val = matrix[row][col]

        if val == target:
            return True
        elif val < target:
            lo = mid + 1
        else:
            hi = mid - 1

    return False


M1 = [
    [1,  3,  5,  7],
    [10, 11, 16, 20],
    [23, 30, 34, 60],
]

cases_74 = [
    ((M1, 7),    True),    # exists, row 0 last
    ((M1, 13),   False),   # not found
    ((M1, 1),    True),    # first element
    ((M1, 60),   True),    # last element
    ((M1, 16),   True),    # middle of matrix
    (([[1]], 1), True),    # 1x1 matrix found
    (([[1]], 2), False),   # 1x1 matrix not found
    (([[1, 3], [5, 7]], 5), True),  # 2x2 matrix
]

test_harness(search_matrix, cases_74)
print("search_matrix defined.")

<a id='9'></a>

## 9. 🍌 Pattern 5 — Search on Answer Space (LC 875 — Koko Eating Bananas)

---

**PROBLEM:** Koko has `piles` of bananas. Guards return in `h` hours. Each hour she eats up to `k` bananas from one pile. Find the minimum `k` so she can finish all piles within `h` hours.

**TRICK:** Binary search on the eating speed `k` from `1` to `max(piles)`. Predicate: can Koko finish all piles in `h` hours at speed `k`? The answer space is sorted — speeds too slow fail, speeds fast enough pass. Find the boundary.

---

**SLOW MOTION TRACE** — `piles=[3,6,7,11]`, `h=8`

```
Answer space: k in [1 .. 11]

lo=1, hi=11
Step 0:  mid=6   can_finish([3,6,7,11], 8, k=6)?
         hours = ceil(3/6)+ceil(6/6)+ceil(7/6)+ceil(11/6) = 1+1+2+2 = 6 <= 8  YES
         hi = 6

Step 1:  lo=1, hi=6  mid=3   can_finish(k=3)?
         hours = ceil(3/3)+ceil(6/3)+ceil(7/3)+ceil(11/3) = 1+2+3+4 = 10 > 8  NO
         lo = 4

Step 2:  lo=4, hi=6  mid=5   can_finish(k=5)?
         hours = ceil(3/5)+ceil(6/5)+ceil(7/5)+ceil(11/5) = 1+2+2+3 = 8 <= 8  YES
         hi = 5

Step 3:  lo=4, hi=5  mid=4   can_finish(k=4)?
         hours = ceil(3/4)+ceil(6/4)+ceil(7/4)+ceil(11/4) = 1+2+2+3 = 8 <= 8  YES
         hi = 4

Terminate: lo==hi==4  → return 4
```

---

**KEY INSIGHT:** When the answer is a number and you can check "does this value work?", binary search on the answer space. The predicate partitions the space into NO...NO...YES...YES — find the leftmost YES.

**TIME:** O(n log(max(piles))) — n to evaluate each candidate, log(max) binary search steps  
**SPACE:** O(1)

In [ ]:
# LC 875 — Koko Eating Bananas
# Time: O(n log(max(piles)))   Space: O(1)

import math

def min_eating_speed(piles, h):
    """
    Find minimum eating speed k so Koko finishes all piles in h hours.
    Time: O(n log(max(piles)))   Space: O(1)

    Slow motion trace (piles=[3,6,7,11], h=8):
      lo=1, hi=11
      step 0: k=6  hours=6 <=8  YES → hi=6
      step 1: k=3  hours=10 >8  NO  → lo=4
      step 2: k=5  hours=8 <=8  YES → hi=5
      step 3: k=4  hours=8 <=8  YES → hi=4
      terminate: lo==hi==4 → return 4
    """

    def can_finish(k):
        # WHY ceil: a partial pile still costs a full hour
        # WHY sum: total hours is sum of hours per pile
        return sum(math.ceil(p / k) for p in piles) <= h

    lo, hi = 1, max(piles)             # WHY 1: min possible speed is 1
                                       # WHY max(piles): eating faster never helps

    while lo < hi:                     # WHY <: left-boundary template, find min valid k
        mid = lo + (hi - lo) // 2

        if can_finish(mid):            # mid is feasible → try smaller (go left)
            hi = mid                   # WHY not mid-1: mid itself might be the answer
        else:
            lo = mid + 1               # mid too slow → need higher speed

    return lo                          # lo == hi: minimum feasible speed


cases_875 = [
    (([3, 6, 7, 11], 8),    4),    # main example
    (([30, 11, 23, 4, 20], 5), 30), # must eat fastest pile in 1 hour
    (([30, 11, 23, 4, 20], 6), 23), # one extra hour helps
    (([1, 1, 1, 1], 4),     1),    # min speed, exactly 4 hours
    (([1000000000], 2),     500000000), # large pile
    (([3, 6, 7, 11], 4),   11),    # only 4 hours, need max speed
]

test_harness(min_eating_speed, cases_875)
print("min_eating_speed defined.")

<a id='10'></a>

## 10. 🗺️ Full Decision Map — All 5 Patterns

```
PATTERN          LC    SIGNAL                            TEMPLATE         KEY MOVE
─────────────────────────────────────────────────────────────────────────────────────────
Classic Search   704   "sorted array, find target"       lo<=hi exact     lo=mid+1 / hi=mid-1
Rotated Search   33    "sorted but rotated"              lo<=hi exact     check which half sorted
Find Min Rotated 153   "rotated, find minimum"           lo<hi boundary   nums[mid]>nums[hi] → lo=mid+1
2D Matrix Search 74    "row-sorted matrix"               lo<=hi exact     row=mid//n, col=mid%n
Answer Space     875   "minimize/maximize a value"       lo<hi boundary   can_do(mid)? hi=mid : lo=mid+1
─────────────────────────────────────────────────────────────────────────────────────────

TEMPLATE CHOICE:
  lo <= hi  →  exact match (return mid or -1)
  lo < hi   →  find boundary (return lo after loop)

HI INITIALIZATION:
  hi = len - 1  →  last valid index (inclusive)
  hi = len      →  one past end (open right boundary for left-boundary template)
  hi = max_val  →  upper bound of answer space

WHEN CAN_DO IS TRUE:
  minimize answer → hi = mid      (keep mid as candidate, discard right)
  maximize answer → lo = mid + 1  (keep going right, discard left)

ROTATED ARRAY PATTERNS (LC 33 vs LC 153):
  LC 33:  compare nums[lo] to nums[mid]  → identify which half sorted
  LC 153: compare nums[mid] to nums[hi]  → find which side has the dip
```

<a id='11'></a>

## 11. 📄 Cheat Sheet

---

### Templates

```python
# EXACT MATCH
lo, hi = 0, len(nums) - 1
while lo <= hi:
    mid = lo + (hi - lo) // 2
    if nums[mid] == target: return mid
    elif nums[mid] < target: lo = mid + 1
    else: hi = mid - 1
return -1

# LEFT BOUNDARY (first occurrence / minimize valid)
lo, hi = 0, len(nums)
while lo < hi:
    mid = lo + (hi - lo) // 2
    if feasible(mid): hi = mid   # shrink right
    else: lo = mid + 1           # shrink left
return lo

# ANSWER SPACE (binary search on values)
lo, hi = min_val, max_val
while lo < hi:
    mid = lo + (hi - lo) // 2
    if can_do(mid): hi = mid
    else: lo = mid + 1
return lo
```

---

### Gotchas

```
OFF-BY-ONE:
  while lo <= hi  uses  hi = mid - 1   (mid already checked)
  while lo < hi   uses  hi = mid       (mid might be the answer)

INFINITE LOOP:
  while lo < hi with lo = mid (not lo = mid+1) → lo never advances → hangs forever
  Rule: at least one of lo, hi must strictly move toward the other every iteration

UNSORTED INPUT:
  Binary search silently returns wrong answers on unsorted data.
  Always verify input is sorted (or has a monotone predicate).

OVERFLOW:
  mid = lo + (hi - lo) // 2   → safe in all languages
  mid = (lo + hi) // 2        → overflows 32-bit int in C/C++ for large arrays
```

---

### Python bisect Module

```python
import bisect
bisect.bisect_left(arr, x)   # leftmost index to insert x (O(log n))
bisect.bisect_right(arr, x)  # rightmost index to insert x (O(log n))
bisect.insort(arr, x)        # insert x keeping arr sorted (O(n) due to shift)
```

---

### Complexity Summary

```
PATTERN                        TIME                   SPACE
─────────────────────────────────────────────────────────────
Classic BS (LC 704)            O(log n)               O(1)
Rotated Search (LC 33)         O(log n)               O(1)
Find Min Rotated (LC 153)      O(log n)               O(1)
2D Matrix Search (LC 74)       O(log(m*n))            O(1)
Answer Space (LC 875)          O(n log(max(piles)))   O(1)
─────────────────────────────────────────────────────────────
```

## Summary

```
                    BINARY SEARCH
                         │
           ┌─────────────┴──────────────┐
     On Indices                   On Values
           │                            │
    ┌──────┴──────┐              Answer Space
  Exact        Boundary           (LC 875)
  Match      (lo < hi)
(lo <= hi)        │
    │        ┌────┴────┐
  LC 704   Left      Right
  LC 33    Bound     Bound
  LC 74    LC 153  (first/last
                   occurrence)


ROTATED ARRAYS:
  Find target  → LC 33   → identify sorted half, check if target in range
  Find minimum → LC 153  → compare mid to hi, go toward the dip

2D MATRIX:
  Flatten → idx = row*n + col
  Unflatten → row = idx//n,  col = idx%n

THE ONE RULE:
  If data is sorted (or has a monotone yes/no predicate),
  binary search will beat linear scan every time.
  O(log n) vs O(n) — for n=10^9, that's 30 ops vs 1 billion.
```

---

*End of Binary Search Master Guide — Sean Edition*